# 05 — Endurance Estimate

Battery capacity → hover endurance with Peukert correction. Sweeps
capacity and payload configuration; marks the 20 % reserve threshold.


In [ ]:
"""Locate the repo root so `import uavdesign` works without `pip install -e .`."""
import sys
from pathlib import Path

REPO_ROOT = Path.cwd()
while not (REPO_ROOT / "uavdesign").is_dir() and REPO_ROOT != REPO_ROOT.parent:
    REPO_ROOT = REPO_ROOT.parent
sys.path.insert(0, str(REPO_ROOT))

PLATFORM_DIR = REPO_ROOT / "platforms" / "multirotor"
FIG_DIR = REPO_ROOT / "outputs" / "figures" / "multirotor"
FIG_DIR.mkdir(parents=True, exist_ok=True)

import yaml
import numpy as np
import matplotlib.pyplot as plt


In [ ]:
from uavdesign import endurance as ed

# Replace this with the value from notebook 03 in a multi-notebook run.
hover_pack_current_a = 30.0  # A, indicative for 6S X6 @ 6.5 kg
pack_voltage_nominal_v = 22.2

capacities_ah = np.linspace(10, 22, 25)
peukert_n = 1.10  # LiPo; use 1.05 for high-quality Li-ion

# Three illustrative payload deltas in additional pack current at hover.
configs = {
    "light  (rgb only)":   0.0,
    "medium (rgb+thermal)": 1.0,
    "heavy  (lidar+altum)": 3.0,
}

fig, ax = plt.subplots(figsize=(7, 4.5))
for label, dI in configs.items():
    minutes = ed.sweep_capacity(
        capacities_ah,
        current_a=hover_pack_current_a + dI,
        nominal_voltage_v=pack_voltage_nominal_v,
        peukert_exponent=peukert_n,
        usable_fraction=0.80,
    )
    ax.plot(capacities_ah, minutes, label=label)

ax.axhline(20, color="grey", ls="--", lw=1, label="20 min target")
ax.set_xlabel("Battery capacity [Ah, 6S]")
ax.set_ylabel("Hover endurance [min, 20 % reserve]")
ax.set_title("Endurance vs capacity, Peukert n = " f"{peukert_n}")
ax.legend()
ax.grid(alpha=0.3)
fig.tight_layout()
fig.savefig(FIG_DIR / "05_endurance_curves.png", dpi=140)
plt.show()


## Endurance surface (capacity x payload)

Useful when reviewing pack options against a payload menu. Replace the
axes with measured per-config hover currents from notebook 04 once
those numbers settle.


In [ ]:
delta_i_a = np.linspace(0, 5, 25)
CC, DD = np.meshgrid(capacities_ah, delta_i_a)
ZZ = np.empty_like(CC)
for i in range(CC.shape[0]):
    ZZ[i, :] = ed.sweep_capacity(
        CC[i, :],
        current_a=hover_pack_current_a + DD[i, 0],
        peukert_exponent=peukert_n,
    )

fig, ax = plt.subplots(figsize=(7, 5))
cs = ax.contourf(CC, DD, ZZ, levels=15, cmap="viridis")
ax.contour(CC, DD, ZZ, levels=[20], colors="white", linewidths=2)
ax.set_xlabel("Battery capacity [Ah]")
ax.set_ylabel("Payload current adder [A]")
ax.set_title("Hover endurance [min]")
fig.colorbar(cs, ax=ax)
fig.tight_layout()
fig.savefig(FIG_DIR / "05_endurance_surface.png", dpi=140)
plt.show()
